# NanoGPT Japanese + TurboQuant

**nanoGPT + TurboQuant KV Cache Compression on Japanese Text**

This notebook demonstrates:
1. Downloading **all** Miyazawa Kenji works from Aozora Bunko (青空文庫) to Google Drive
2. Character-level Japanese tokenizer on a large corpus
3. Training a GPT-2 small equivalent on complete Miyazawa Kenji literature (block_size=1024)
4. Inference with stop sequences (。sentence boundaries)
5. Standard vs TurboQuant comparison at longer context

**Training corpus:** 宮沢賢治 全作品 from 青空文庫 (cached on Google Drive)

**All data & checkpoints are saved to Google Drive** at `colab/nano_gpt/data/` to persist across sessions.

**References:**
- [nanoGPT](https://github.com/karpathy/nanoGPT) - Andrey Karpathy
- [TurboQuant](https://arxiv.org/abs/2504.19874) - ICLR 2026
- [青空文庫](https://www.aozora.gr.jp/) - Public domain Japanese literature

## 0. Setup & Google Drive Mount

In [ ]:
import os, sys

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Google Drive paths for persistent storage
DRIVE_BASE = '/content/drive/MyDrive/colab/nano_gpt'
DRIVE_DATA_DIR = os.path.join(DRIVE_BASE, 'data')
DRIVE_RAW_DIR = os.path.join(DRIVE_DATA_DIR, 'aozora_raw')  # raw downloaded texts
DRIVE_PREPARED_DIR = os.path.join(DRIVE_DATA_DIR, 'aozora')  # train.bin, val.bin, meta.pkl
DRIVE_OUT_DIR = os.path.join(DRIVE_BASE, 'out-aozora')       # checkpoints

for d in [DRIVE_DATA_DIR, DRIVE_RAW_DIR, DRIVE_PREPARED_DIR, DRIVE_OUT_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Google Drive data dir: {DRIVE_DATA_DIR}')
print(f'Google Drive output dir: {DRIVE_OUT_DIR}')

# Clone the repository (use absolute path to avoid nesting on re-run)
REPO_DIR = '/content/NanoGPT_with_TurboQuant'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/3t14/NanoGPT_with_TurboQuant.git {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

# Symlink data and output dirs to Google Drive for persistence
for local, remote in [('data/aozora', DRIVE_PREPARED_DIR), ('out-aozora', DRIVE_OUT_DIR)]:
    parent = os.path.dirname(local)
    if parent:
        os.makedirs(parent, exist_ok=True)
    if os.path.islink(local):
        os.unlink(local)
    elif os.path.exists(local):
        import shutil
        shutil.rmtree(local)
    os.symlink(remote, local)
    print(f'Symlinked {local} -> {remote}')

!pip install requests numpy -q

In [ ]:
import torch
import numpy as np
import math
import time
import pickle
import re

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB')

## 1. Download & Prepare Japanese Text

Download **all** works by 宮沢賢治 from Aozora Bunko (青空文庫) using the official CSV index.
Previously downloaded texts are cached on Google Drive at `colab/nano_gpt/data/aozora_raw/` and skipped on re-run.

This provides a much larger training corpus (~数十万 characters) to reduce overfitting.

In [ ]:
import requests
import zipfile
import io
import csv
import glob
import re
import hashlib

def download_aozora_zip(url):
    """Download and extract text from an Aozora Bunko zip URL."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for name in zf.namelist():
            if name.endswith('.txt'):
                raw = zf.read(name)
                for enc in ['shift_jis', 'utf-8', 'cp932']:
                    try:
                        return raw.decode(enc)
                    except (UnicodeDecodeError, LookupError):
                        continue
    raise RuntimeError('Could not decode text')

def clean_aozora(text):
    """Clean Aozora Bunko formatting."""
    lines = text.split('\n')
    start = 0
    for i, line in enumerate(lines):
        if '---' in line and i > 5:
            start = i + 1
            break
    end = len(lines)
    for i in range(len(lines) - 1, 0, -1):
        if lines[i].startswith('底本') or lines[i].startswith('青空文庫'):
            end = i
            break
    text = '\n'.join(lines[start:end])
    text = re.sub(r'《[^》]*》', '', text)
    text = text.replace('｜', '')
    text = re.sub(r'［＃[^］]*］', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# --- Fetch the Aozora Bunko CSV index to find all 宮沢賢治 works ---
print('Fetching Aozora Bunko CSV index...')
CSV_URL = 'https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip'
csv_cache_path = os.path.join(DRIVE_RAW_DIR, 'list_person_all_extended_utf8.csv')

if not os.path.exists(csv_cache_path):
    resp = requests.get(CSV_URL, timeout=60)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for name in zf.namelist():
            if name.endswith('.csv'):
                with open(csv_cache_path, 'wb') as f:
                    f.write(zf.read(name))
                break
    print(f'  CSV index saved to Google Drive ({os.path.getsize(csv_cache_path) / 1e6:.1f} MB)')
else:
    print(f'  CSV index already cached on Google Drive')

# Parse CSV to find all 宮沢賢治 works with zip download URLs
kenji_works = []
with open(csv_cache_path, 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get('姓', '') == '宮沢' and row.get('名', '') == '賢治':
            zip_url = row.get('テキストファイルURL', '')
            if zip_url and zip_url.endswith('.zip'):
                title = row.get('作品名', '不明')
                kenji_works.append((title, zip_url))

print(f'Found {len(kenji_works)} Miyazawa Kenji works with zip downloads')

# --- Download all works (skip if already cached on Google Drive) ---
all_text = ''
success_count = 0
skip_count = 0
fail_count = 0

for title, zip_url in sorted(kenji_works, key=lambda x: x[0]):
    # Use URL hash as stable filename for cache
    safe_name = hashlib.md5(zip_url.encode()).hexdigest()
    cache_path = os.path.join(DRIVE_RAW_DIR, f'{safe_name}.txt')

    if os.path.exists(cache_path):
        # Already downloaded - load from Google Drive cache
        with open(cache_path, 'r', encoding='utf-8') as f:
            cleaned = f.read()
        if len(cleaned) > 100:
            all_text += f'\n\n{cleaned}'
            skip_count += 1
            continue

    # Download and clean
    try:
        raw = download_aozora_zip(zip_url)
        cleaned = clean_aozora(raw)
        if len(cleaned) < 100:
            print(f'  {title}: too short ({len(cleaned)} chars), skipping')
            fail_count += 1
            continue

        # Save cleaned text to Google Drive cache
        with open(cache_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)

        all_text += f'\n\n{cleaned}'
        success_count += 1
        print(f'  ✓ {title}: {len(cleaned):,} chars')
    except Exception as e:
        print(f'  ✗ {title}: FAILED - {e}')
        fail_count += 1

all_text = all_text.strip()
print(f'\n=== Corpus Summary ===')
print(f'Total corpus: {len(all_text):,} characters')
print(f'New downloads: {success_count}, Cached (skipped): {skip_count}, Failed: {fail_count}')
print(f'Raw texts cached at: {DRIVE_RAW_DIR}')

In [ ]:
# Preview the text
print('=== First 500 characters ===')
print(all_text[:500])
print('\n=== Last 300 characters ===')
print(all_text[-300:])

## 2. Build Character-Level Japanese Tokenizer

Same approach as nanoGPT's Shakespeare tokenizer, but for Japanese characters.
A single novel typically uses 2000-3000 unique characters.

In [ ]:
# Build character-level tokenizer
chars = sorted(list(set(all_text)))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
print(f'Sample characters: {chars[:30]}...')

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return ''.join([itos[i] for i in l])

# Verify roundtrip
test = '銀河鉄道の夜'
assert decode(encode(test)) == test
print(f'Encode/decode test passed: "{test}" -> {encode(test)} -> "{decode(encode(test))}"')

# Create train/val split
n = len(all_text)
train_data = all_text[:int(n * 0.9)]
val_data = all_text[int(n * 0.9):]

train_ids = np.array(encode(train_data), dtype=np.uint16)
val_ids = np.array(encode(val_data), dtype=np.uint16)

print(f'Train: {len(train_ids):,} tokens')
print(f'Val: {len(val_ids):,} tokens')

# Save to data/aozora/ (symlinked to Google Drive)
data_dir = os.path.join('data', 'aozora')
os.makedirs(data_dir, exist_ok=True)

train_ids.tofile(os.path.join(data_dir, 'train.bin'))
val_ids.tofile(os.path.join(data_dir, 'val.bin'))

meta = {'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}
with open(os.path.join(data_dir, 'meta.pkl'), 'wb') as f:
    pickle.dump(meta, f)

print(f'\nSaved to {data_dir}/ (Google Drive: {DRIVE_PREPARED_DIR})')
print(f'  train.bin: {os.path.getsize(os.path.join(data_dir, "train.bin")) / 1e6:.2f} MB')
print(f'  val.bin:   {os.path.getsize(os.path.join(data_dir, "val.bin")) / 1e6:.2f} MB')

## 3. Train on Japanese Text

Train a GPT with **block_size=1024** and a larger model (GPT-2 small equivalent).
With the full Miyazawa Kenji corpus, overfitting should be significantly reduced.

Checkpoints are saved to **Google Drive** (`out-aozora/`) so training can be resumed across sessions.

| Parameter | Value | Note |
|-----------|-------|------|
| block_size | 1024 | 4x longer than Shakespeare demo |
| batch_size | 16 | Fits in T4 with larger model |
| gradient_accumulation_steps | 4 | Effective batch = 64 |
| **n_layer / n_head / n_embd** | **12 / 12 / 768** | **GPT-2 small equivalent (~88M params)** |
| dropout | 0.2 | Standard regularization |
| max_iters | 3000 | Sufficient training with full corpus |

In [ ]:
# Train with block_size=1024 and GPT-2 small equivalent model
# Checkpoints saved to out-aozora/ (symlinked to Google Drive)
!python train.py \
    --dataset=aozora \
    --out_dir=out-aozora \
    --device={device} \
    --compile=False \
    --block_size=1024 \
    --batch_size=16 \
    --gradient_accumulation_steps=4 \
    --n_layer=12 \
    --n_head=12 \
    --n_embd=768 \
    --dropout=0.2 \
    --learning_rate=6e-4 \
    --max_iters=3000 \
    --eval_interval=300 \
    --eval_iters=100 \
    --lr_decay_iters=3000 \
    --min_lr=6e-5 \
    --warmup_iters=200 \
    --always_save_checkpoint=True \
    --use_turboquant=True \
    --turboquant_bits=3

print(f'\nCheckpoint saved to Google Drive: {DRIVE_OUT_DIR}')

## 4. Generate Japanese Text with Stop Sequences

Generate text that stops at natural Japanese sentence boundaries (。) or paragraph breaks (\n\n).

In [ ]:
from model import GPT, GPTConfig

# Load trained model (from Google Drive via symlink)
out_dir = 'out-aozora'
ckpt_path = f'{out_dir}/ckpt.pt'

if not os.path.exists(ckpt_path):
    print(f'ERROR: {ckpt_path} not found!')
    print(f'Check Google Drive: {DRIVE_OUT_DIR}')
    print('Training may have failed (OOM?). Check the training cell output above.')
    raise FileNotFoundError(ckpt_path)

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model_args = ckpt['model_args']
model_args['use_turboquant'] = True
model_args['turboquant_bits'] = 3

config = GPTConfig(**model_args)
model = GPT(config)

state_dict = ckpt['model']
for k in list(state_dict.keys()):
    if k.startswith('_orig_mod.'):
        state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)
model.load_state_dict(state_dict)
model.eval()
model.to(device)

# Load tokenizer (from Google Drive via symlink)
with open('data/aozora/meta.pkl', 'rb') as f:
    meta = pickle.load(f)
stoi, itos = meta['stoi'], meta['itos']
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(f'Model loaded: {model.get_num_params()/1e6:.2f}M parameters')
print(f'Config: {config.n_layer}L, {config.n_head}H, {config.n_embd}D, block_size={config.block_size}')
print(f'Vocab size: {config.vocab_size}')
print(f'Checkpoint from: {DRIVE_OUT_DIR}')

In [ ]:
def generate_with_stop(prompt, max_tokens=1024, use_turboquant=False,
                       temperature=0.9, top_k=200,
                       stop_sequences=None, seed=42):
    """
    Generate text with stop sequence support.

    Args:
        stop_sequences: list of strings to stop at (e.g., ['。', '\n\n'])
                        If None, defaults to ['。']
        temperature: higher values = more creative/diverse (default 0.9)
    """
    if stop_sequences is None:
        stop_sequences = ['。', '\n\n']

    start_ids = encode(prompt)
    x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

    torch.manual_seed(seed)
    t0 = time.time()
    with torch.no_grad():
        y = model.generate(x, max_tokens, temperature=temperature, top_k=top_k,
                           use_turboquant=use_turboquant)
    elapsed = time.time() - t0

    full_text = decode(y[0].tolist())
    generated = full_text[len(prompt):]
    n_generated = len(generated)

    # Find first stop sequence
    stop_pos = len(generated)
    matched_stop = None
    for stop in stop_sequences:
        idx = generated.find(stop)
        if idx >= 0 and idx + len(stop) < stop_pos:
            stop_pos = idx + len(stop)
            matched_stop = repr(stop)

    trimmed = generated[:stop_pos]
    tok_per_sec = n_generated / elapsed

    return {
        'prompt': prompt,
        'generated': trimmed,
        'full_generated': generated,
        'stopped_by': matched_stop or f'max_tokens ({max_tokens})',
        'elapsed': elapsed,
        'tok_per_sec': tok_per_sec,
        'n_tokens': n_generated,
    }

In [ ]:
# Generate and compare: Standard vs TurboQuant
prompts = [
    'ジョバンニは、',
    'カムパネルラが、',
    '夜の軽便鉄道は、',
]

max_tokens = 1024

for prompt in prompts:
    print('=' * 60)
    print(f'Prompt: 「{prompt}」')
    print('=' * 60)

    # Standard
    r_std = generate_with_stop(prompt, max_tokens=max_tokens, use_turboquant=False)
    print(f'\n--- Standard ({r_std["tok_per_sec"]:.0f} tok/s) ---')
    print(f'{prompt}{r_std["generated"]}')
    print(f'  [Stopped by: {r_std["stopped_by"]}]')

    # TurboQuant
    r_tq = generate_with_stop(prompt, max_tokens=max_tokens, use_turboquant=True)
    print(f'\n--- TurboQuant 3-bit ({r_tq["tok_per_sec"]:.0f} tok/s) ---')
    print(f'{prompt}{r_tq["generated"]}')
    print(f'  [Stopped by: {r_tq["stopped_by"]}]')
    print()

In [ ]:
# Generate longer passages (stop at paragraph boundary)
prompt = 'ジョバンニは、'

print('=== Long generation (stop at paragraph break) ===')
print(f'Prompt: 「{prompt}」\n')

r = generate_with_stop(
    prompt,
    max_tokens=1024,
    use_turboquant=True,
    stop_sequences=['\n\n'],
    temperature=0.7,
)
print(f'{prompt}{r["generated"]}')
print(f'\n[{r["n_tokens"]} tokens | {r["elapsed"]:.2f}s | {r["tok_per_sec"]:.0f} tok/s | Stopped by: {r["stopped_by"]}]')

## 5. Quality Analysis: Bit-width Comparison

Compare text quality at different TurboQuant bit-widths.

In [ ]:
prompt = 'カムパネルラは、'

for bits in [None, 4, 3, 2]:
    label = f'TQ-{bits}bit' if bits else 'Standard'
    use_tq = bits is not None

    if use_tq:
        for block in model.transformer.h:
            block.attn.turboquant_bits = bits
            block.attn._tq_compressor = None

    r = generate_with_stop(prompt, max_tokens=512, use_turboquant=use_tq,
                           stop_sequences=['。'])

    print(f'--- {label} ({r["tok_per_sec"]:.0f} tok/s) ---')
    print(f'{prompt}{r["generated"]}')
    print(f'  [Stopped by: {r["stopped_by"]}]')
    print()

# Reset to 3-bit
for block in model.transformer.h:
    block.attn.turboquant_bits = 3
    block.attn._tq_compressor = None

## 6. Long Context Compression Benchmarks

TurboQuant's memory savings are most significant with longer sequences.
Compare compression at different context lengths.

In [ ]:
from turboquant import TurboQuantKVCompressor

head_dim = config.n_embd // config.n_head
n_heads = config.n_head
n_layers = config.n_layer

print(f'Model: {n_layers} layers, {n_heads} heads, head_dim={head_dim}')
print(f'Block size: {config.block_size}\n')

print(f'{"Seq Len":>8} | {"Bits":>4} | {"FP16 KV":>10} | {"TQ KV":>10} | {"Ratio":>6} | {"Saved":>8}')
print('-' * 65)

for T in [256, 512, 1024]:
    for bits in [3, 4]:
        comp = TurboQuantKVCompressor(head_dim, bits=bits)
        info = comp.memory_savings_info(T, n_heads)
        fp16_kb = info['fp16_bits'] / 8 / 1024 * n_layers * 2
        tq_kb = info['compressed_bits'] / 8 / 1024 * n_layers * 2
        saved_kb = fp16_kb - tq_kb
        print(f'{T:>8} | {bits:>4} | {fp16_kb:>8.1f} KB | {tq_kb:>8.1f} KB | {info["compression_ratio"]:>5.1f}x | {saved_kb:>6.1f} KB')

In [ ]:
import matplotlib.pyplot as plt

# Speed benchmark at different sequence lengths
bench_device = device
D = head_dim
H = n_heads

seq_lens = [128, 256, 512, 1024]
results = {}

for T in seq_lens:
    B = 2
    k = torch.randn(B, H, T, D, device=bench_device)
    v = torch.randn(B, H, T, D, device=bench_device)
    q = torch.randn(B, H, T, D, device=bench_device)

    comp = TurboQuantKVCompressor(D, bits=3, device=bench_device)

    ck = comp.compress_keys(k)
    cv = comp.compress_values(v)
    _ = comp.attention_scores(q, ck)
    _ = comp.decompress_values(cv)
    if bench_device == 'cuda':
        torch.cuda.synchronize()

    n_iter = 20

    t0 = time.time()
    for _ in range(n_iter):
        ck = comp.compress_keys(k)
        cv = comp.compress_values(v)
        _ = comp.attention_scores(q, ck)
        _ = comp.decompress_values(cv)
    if bench_device == 'cuda':
        torch.cuda.synchronize()
    t_tq = (time.time() - t0) / n_iter * 1000

    t0 = time.time()
    for _ in range(n_iter):
        s = torch.matmul(q, k.transpose(-2, -1)) * (1.0 / math.sqrt(D))
        _ = torch.matmul(torch.softmax(s, dim=-1), v)
    if bench_device == 'cuda':
        torch.cuda.synchronize()
    t_std = (time.time() - t0) / n_iter * 1000

    info = comp.memory_savings_info(T, H)
    results[T] = {'t_tq': t_tq, 't_std': t_std, 'ratio': info['compression_ratio']}
    print(f'T={T:>5}: TQ={t_tq:.1f}ms, Std={t_std:.1f}ms, Compression={info["compression_ratio"]:.1f}x')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ts = list(results.keys())
tq_times = [results[t]['t_tq'] for t in ts]
std_times = [results[t]['t_std'] for t in ts]
ratios = [results[t]['ratio'] for t in ts]

ax1.plot(ts, std_times, 'b-o', label='Standard', linewidth=2)
ax1.plot(ts, tq_times, 'r-o', label='TurboQuant 3-bit', linewidth=2)
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Attention Computation Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

fp16_mem = [t * D * H * 2 * 16 / 8 / 1024 for t in ts]
tq_mem = [fp16_mem[i] / ratios[i] for i in range(len(ts))]

ax2.bar([str(t) for t in ts], fp16_mem, alpha=0.7, label='FP16', color='steelblue')
ax2.bar([str(t) for t in ts], tq_mem, alpha=0.7, label='TQ 3-bit', color='coral')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('KV Cache Size (KB)')
ax2.set_title('KV Cache Memory per Layer')
ax2.legend()

plt.suptitle('TurboQuant Benefits Scale with Sequence Length', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary

### Japanese Text Generation with TurboQuant

| Setting | Compression | Quality |
|---------|-------------|--------|
| Standard (FP16) | 1.0x | Baseline |
| TurboQuant 4-bit | ~3.8x | Imperceptible difference |
| TurboQuant 3-bit | ~4.9x | Near-lossless |
| TurboQuant 2-bit | ~6.5x | Noticeable degradation |

### Key Observations

- **block_size=1024** enables longer, more coherent Japanese passages
- **Stop sequences** (。, \n\n) produce natural sentence/paragraph endings
- **Character-level tokenization** works well for Japanese (vocab ~2000-3000)
- **TurboQuant 3-bit** is the sweet spot: ~5x compression with minimal quality loss
- **Memory savings scale linearly** with sequence length: longer contexts benefit more